### Pseudonymization Strategy & GDPR Compliance

To secure the dataset, we applied cryptographic hashing (pseudonymization) to direct identifiers such as `.ssn` `.email` and `.full_name`. 

**Why we chose this approach and why it is legally sufficient:**

* **Preserving Data Utility:** Unlike full data deletion or simple masking, pseudonymization protects user privacy while allowing the Data Science team to track unique applicant records. This is essential for accurately calculating fairness metrics and auditing the credit model.
* **Data Minimization & Security (Art. 5 & Art. 32):** By transforming raw PII into hashed values, we ensure that sensitive personal data is no longer stored without protection, preventing exposure in plain text to analysts or downstream systems.
* **Right to Erasure (Art. 17):** This approach elegantly handles the "Right to be Forgotten". By using a secret cryptographic key (a "salt") during the hashing process, we can simply delete the key when a user requests data erasure. This process, known as *crypto-shredding*, irreversibly turns the pseudonymized data into fully anonymized data, It satisfies GDPR Article 17 requirements perfectly, without forcing us to destroy valuable historical statistical data.

In [ ]:
import pandas as pd
import hashlib
import json

# 1. Load the nested JSON data
with open('../data/raw_credit_applications.json', 'r') as file:
    data = json.load(file)

df = pd.json_normalize(data)

# 2. Define the pseudonymization function
SALT = "NovaCred_Secure_2026!"

def pseudonymize_pii(value):
    if pd.isna(value) or value == "":
        return value
    
    # Combine the value with the pseudonym and encode
    salted_value = str(value) + SALT
    
    # Return the SHA-256 hexadecimal hash
    return hashlib.sha256(salted_value.encode()).hexdigest()

# 3. Apply the function to the PII columns
# Targeting the nested fields from the schema
df['applicant_info.ssn_hashed'] = df['applicant_info.ssn'].apply(pseudonymize_pii)
df['applicant_info.email_hashed'] = df['applicant_info.email'].apply(pseudonymize_pii)
df['applicant_info.full_name_hashed'] = df['applicant_info.full_name'].apply(pseudonymize_pii)
# 4. Drop the original raw PII columns to ensure data minimization
df = df.drop(columns=['applicant_info.ssn', 'applicant_info.email', 'applicant_info.full_name'])

# 5. Verify the transformation
print(df[['_id', 'applicant_info.ssn_hashed', 'applicant_info.email_hashed','applicant_info.full_name_hashed']].head())

       _id                          applicant_info.ssn_hashed  \
0  app_200  9196537eb3af63e06878de29f8e1c7d3897791fe6032e1...   
1  app_037  f7e04e392a2676c1008fc9dc21750b3176c57c5eed8809...   
2  app_215  3263f19e841649667a994a137d9d2444feae15e1b50417...   
3  app_024  496b136660dfbe2bf483a50420f917e89e49e666a0ae79...   
4  app_184  5f688fde89897de7828b5eddf85be3ab066b2d9a5066da...   

                         applicant_info.email_hashed  \
0  43816fd1eb76c24869e6e312699cda3d95ff057122fbfb...   
1  5f4a8dd0e06f28d96333869faf36bc4b38ba9727c68526...   
2  63be22cf9717bb8f86a2800acae73c429d08b8246ea6a3...   
3  807aef6fb2d50f66c53a38597b8e437b955923c8636547...   
4  6bead4f1f8650f583ff89c06a5697b6aab4a8c20db519c...   

                     applicant_info.full_name_hashed  
0  070c9881585996a6be0283cff64d93979b3a82736e602e...  
1  5818e748b1730f6f2e23c72eb1db5b19a09edf7207351e...  
2  eb51c1b818060437741f3718afcc89d6aa27c7c309a7c7...  
3  a5a462d3cab268cf103c86d825b6e69a5c01756634f281...